# 01 — Map the buildings on one farm, step by step

Every step below calls the same functions `scripts/map_buildings.py` runs over all 417 farms (`geo_anom/task1/`). Nothing here is a re-implementation — if a result looks wrong in this notebook, it is wrong in the full run too.

**Steps:** pick a farm → load its tile → look at the pixels → run the U-Net → turn the mask into shapes → apply the shape filter → box the buildings → check them yourself → export a KMZ (one marker style per animal type).

Run cells top to bottom. Change `FARM` in step 1 and re-run to check another farm.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib inline
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)

from geo_anom.task1 import tiles, detect, viz
from geo_anom.task1.species import species_group, GROUPS

## 1. Pick a farm

The manifest is the list of registry permits with a downloaded tile (2 km × 2 km around the permit point). `find_site` does a case-insensitive substring match.

Registry permit points are often **400–860 m** from the real barns (2026-09-23 imagery review), so the barns are not necessarily in the centre of the tile.

In [ ]:
manifest = tiles.load_manifest()
print(len(manifest), "sites in manifest")
display(pd.Series([s["animal_type"] for s in manifest]).value_counts().rename("farms by registry animal_type"))

In [ ]:
FARM = "Sheng Lin"          # try also: "Alan C. Eck", "Lester C. Jones" (dairy), "Panora" (beef), "Hite Farms"

site = tiles.find_site(manifest, FARM)
print(json.dumps(site, indent=2))
print("species group:", species_group(site["animal_type"]))

## 2. Load the tile and look at it

4 bands: R, G, B, NIR, at 1 m/pixel (Planetary Computer NAIP, resampled from 0.3 m — the model was trained on 1 m). The false-colour view puts NIR in red: vegetation glows red, roofs and pavement stay grey/cyan, water goes black.

In [ ]:
img, meta = tiles.read_tile(tiles.tile_path(site))
print("shape (H, W, bands):", img.shape, "| dtype:", img.dtype)
print("CRS:", meta["crs"], "| pixel size (m):", round(meta["res_m"], 3))
print("imagery date:", site.get("naip_datetime"))
print("band means (R, G, B, NIR):", img.reshape(-1, 4).mean(0).round(1))
viz.show_tile(img, site["farm_name"]);

## 3. Run the U-Net

Robinson et al. (2022) checkpoint, trained on Delmarva poultry houses (Soroka & Duren labels). It outputs, per pixel, the probability of "building". The image is processed in 256-px chips with 32-px overlap, blended — the same tiling Microsoft used.

**Check:** do the bright areas in the probability map sit on the barns? Are there barns with no probability at all (that's a model miss — no filter can recover it)?

In [ ]:
model, device = detect.load_model()
print("device:", device)

scores = detect.predict_class_scores(model, device, img)
prob, mask = scores[1], detect.scores_to_mask(scores)
viz.show_model_output(img, prob, mask);

## 4. Mask → shapes

The mask is closed with an 11×11 kernel first (joins a house split by a dark roof ridge into one piece), then each connected blob becomes a polygon in the tile's UTM coordinates (metres).

In [ ]:
polys = detect.mask_to_polygons(mask, meta["transform"])
print(len(polys), "candidate shapes")

## 5. Shape filter (Tulbure et al. 2024, recalibrated for Maryland)

Each candidate's minimum rotated rectangle gives length and width. A candidate is kept only if **all** of these hold (beef farms have looser length/aspect minimums):

In [ ]:
display(pd.Series(detect.filter_thresholds(site["animal_type"]), name="threshold"))

In [ ]:
result = detect.detect_site(site, model, device)   # = steps 2-5 in one call, as the full run does
assert (result.mask == mask).all()                    # same thing you just looked at

table = pd.DataFrame([{
    "#": n,
    "kept": c["kept"],
    "area_m2": round(c["stats"]["area_m2"]),
    "length_m": round(c["stats"]["length_m"], 1),
    "width_m": round(c["stats"]["width_m"], 1),
    "aspect": round(c["stats"]["aspect"], 2),
    "mean_prob": c["feature"]["properties"]["mean_prob"],
    "why rejected": "; ".join(c["reasons"]),
} for n, c in enumerate(result.candidates)])
print(f"{len(result.kept)} kept, {len(result.rejected)} rejected")
table.sort_values(["kept", "area_m2"], ascending=[False, False])

## 6. Box the buildings

Green = kept, red dashed = rejected. The thick box is the minimum rotated rectangle the filter measured; the thin line is the actual mask outline. Numbers match the `#` column above.

In [ ]:
viz.show_boxes(result);

Zoom into any candidate by its number — true colour and false colour side by side. (`viz.show_boxes(result, zoom=None)` shows the whole tile.)

**Look at how much of the roof the outline covers.** On many houses the mask covers only part of the roof (e.g. Sheng Lin #0: 108 × 18 m outline on a visibly longer, wider house). Detected floor area — used to split a farm's N/P across its barns — is therefore an underestimate, and not uniformly so.

In [ ]:
if result.candidates:
    viz.show_candidate(result, 0);

## 7. Check it yourself

Go through the boxes and record what you see. This is the part no script can do for you. For each farm:

| | count |
|---|---|
| **TP** — kept box on a real animal house | |
| **FP** — kept box on something else (house, shed, road, field) | |
| **FN** — real animal house with no kept box (look for red boxes on real barns, and barns with no box at all) | |

Then precision = TP / (TP + FP), recall = TP / (TP + FN). Fill the dict below and it is appended to `notebooks/output/my_audit.csv`, so the audit builds up farm by farm.

In [ ]:
MY_COUNTS = {"TP": None, "FP": None, "FN": None, "notes": ""}   # <- fill in, then run

if MY_COUNTS["TP"] is not None:
    out = ROOT / "notebooks/output/my_audit.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    row = pd.DataFrame([{"farm_name": site["farm_name"], "animal_type": site["animal_type"],
                         "kept": len(result.kept), **MY_COUNTS,
                         "checked_at": pd.Timestamp.now().isoformat(timespec="minutes")}])
    row.to_csv(out, mode="a", header=not out.exists(), index=False)
    display(pd.read_csv(out))

## 8. Compare with the committed baseline

`data/processed/detections/full_registry_unet_tulbure_detections.geojson` is the file every reported number (R² = 0.655, 99.7% precision) was computed from. It has one feature per *tile detection*, so overlapping neighbour tiles count the same barn more than once.

In [ ]:
base = json.loads((ROOT / "data/processed/detections/full_registry_unet_tulbure_detections.geojson").read_text())["features"]
n_base = sum(f["properties"]["farm_name"] == site["farm_name"] for f in base)
print(f"committed baseline: {n_base} | today: {len(result.kept)}")
if n_base != len(result.kept):
    print("DIFFERENT -- see docs/research_log.md 2026-09-21 'drift' entry; check which one is right by eye above.")

## 9. Several farms → de-duplicate → KMZ by animal type

Neighbouring tiles overlap, so a barn can be detected from several farms' tiles. `deduplicate` keeps one copy (highest model confidence) and assigns it to the **nearest permit point** — check `assigned_distance_m`: hundreds of metres means the attribution is a guess.

The KMZ has one folder and marker per species group (circle = broiler, diamond = layers/turkeys/ducks, triangle = swine, square = beef/dairy), rejected candidates in a hidden folder with the reason in the balloon, and registry pins. Open it in Google Earth.

In [ ]:
from geo_anom.task1.merge import deduplicate
from geo_anom.task1.kmz import write_kmz

FARMS = [FARM, "Lester C. Jones", "Panora", "Vy Tran"]   # mix species to see the different markers

results = [detect.detect_site(tiles.find_site(manifest, f), model, device) for f in FARMS]
kept = [f for r in results for f in r.features()]
rejected = [c["feature"] for r in results for c in r.rejected]
buildings = deduplicate(kept, manifest)
print(f"{len(kept)} kept detections -> {len(buildings)} unique buildings")
display(buildings.groupby(["assigned_farm", "species_group"]).agg(
    buildings=("building_id", "size"), area_m2=("area_m2", "sum"),
    max_assign_dist_m=("assigned_distance_m", "max"), seen_twice_or_more=("n_detections", lambda s: int((s > 1).sum()))))

In [ ]:
features = json.loads(buildings.to_json(drop_id=True))["features"] + rejected
kmz_path = write_kmz(features, ROOT / "notebooks/output/check_farms.kmz",
                     sites=[r.site for r in results], title="Notebook check")
print("KMZ ->", kmz_path)